# Exploratory Data Analysis (EDA) - NYC TLC (Yellow Taxi)

Este notebook tem como objetivo realizar uma Análise Exploratória de Dados (EDA) profunda sobre TODAS as colunas e todos os dados de corridas de táxi em Nova York ingeridos na camada Bronze.


In [ ]:
from pyspark.sql.functions import col, sum, isnan, when, min, max
from functools import reduce

In [15]:
# Verifica se a variável 'spark' já existe (como acontece na interface Web do Databricks)
if 'spark' not in locals() and 'spark' not in globals():
    from databricks.connect import DatabricksSession
    # Usa o perfil autenticado da sua extensão do VS Code
    spark = DatabricksSession.builder.getOrCreate()
    print("Conectado ao Databricks remoto via Databricks Connect! 🚀")
else:
    print("Sessão nativa do Databricks identificada.")


Sessão nativa do Databricks identificada.


## 1. Carregamento e Unificação dos Dados (Camada Bronze)
Como a agência TLC mudou o schema entre os meses de 2023 (alterando nomes como `Airport_fee` vs `airport_fee` e tipos como `DOUBLE` vs `BIGINT`), fazemos a leitura mês a mês, normalizamos os esquemas para manter TODAS as colunas e unimos com `unionByName`.

In [3]:
schema_esperado = {
    "vendorid": "long",
    "tpep_pickup_datetime": "timestamp",
    "tpep_dropoff_datetime": "timestamp",
    "passenger_count": "double",
    "trip_distance": "double",
    "ratecodeid": "double",
    "store_and_fwd_flag": "string",
    "pulocationid": "long",
    "dolocationid": "long",
    "payment_type": "long",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "improvement_surcharge": "double",
    "total_amount": "double",
    "congestion_surcharge": "double",
    "airport_fee": "double"
}

dfs = []
for mes in range(1, 6):
    path_mes = f"s3://ifood-case-lucas-datalake-762131618299-us-east-2-an/bronze/nyc_tlc/yellow/ano=2023/mes={str(mes).zfill(2)}/*.parquet"
    df_mes = spark.read.parquet(path_mes)
    
    # 1. Resolver Case Sensitivity: Forçar todas as colunas originais para minúsculo
    df_mes = df_mes.toDF(*[c.lower() for c in df_mes.columns])
    
    # 2. Aplicar o Schema Fixo Esperado (Cast dinâmico para blindar contra Type Drift)
    for col_name, col_type in schema_esperado.items():
        if col_name in df_mes.columns:
            df_mes = df_mes.withColumn(col_name, col(col_name).cast(col_type))
            
    dfs.append(df_mes)

# União de todos os meses de forma blindada contra Schema Drift futuro
df_eda = reduce(lambda d1, d2: d1.unionByName(d2, allowMissingColumns=True), dfs)

print(f"Total de registros: {df_eda.count():,}")
print(f"Total de colunas exploradas: {len(df_eda.columns)}")
df_eda.printSchema()

Total de registros: 16,186,386
Total de colunas exploradas: 19
root
 |-- vendorid: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecodeid: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pulocationid: long (nullable = true)
 |-- dolocationid: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



## 2. Análise Exploratória (EDA)

### 2.1 Verificação de Nulos
Verificando dados faltantes em TODAS as colunas do dataset bruto.

In [4]:
from pyspark.sql.types import DoubleType, FloatType

exprs = []
for f in df_eda.schema.fields:
    c = f.name
    # isnan() só existe nativamente para tipos Float e Double (NaN = Not a Number)
    if isinstance(f.dataType, (DoubleType, FloatType)):
        expr = sum(when(col(c).isNull() | isnan(col(c)), 1).otherwise(0)).alias(c)
    else:
        expr = sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    exprs.append(expr)

df_nulls = df_eda.select(*exprs)
display(df_nulls)

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,0,0,0,428665,0,428665,428665,0,0,0,0,0,0,0,0,0,0,428665,428665


### 2.2 Estatísticas Descritivas Gerais
Entendendo as distribuições, mínimas e máximas de todas as colunas numéricas e financeiras.

In [5]:
display(df_eda.summary())

,summary,vendorid,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,count,16186386,15757721,16186386,15757721,15757721,16186386,16186386,16186386,16186386,16186386,16186386,16186386,16186386,16186386,16186386,15757721,15757721
1,mean,1.7308431295287285,1.3617580232572972,4.021714984431687,1.567152953145953,None,165.75723765638605,164.25596856518806,1.1892137627262813,18.984075327871626,1.5846742509415053,0.48733660497161174,3.4795487800685563,0.5602047696153654,0.9812967082336321,27.83854981703999,2.2743567581885733,0.1189021718305585
2,stddev,0.4490355016836576,0.8914096737459007,253.83256560812282,6.963272126475176,None,64.0538486321142,69.87910236882148,0.5360988165605481,18.38060168323733,1.8305524152337924,0.10312126647772318,3.9821134219684478,2.116199645357476,0.1886379869372363,22.940227504361847,0.7760047905101902,0.4090645461790339
3,min,1,0.0,0.0,1.0,N,1,1,0,-959.9,-7.5,-0.5,-411.0,-73.3,-1.0,-982.95,-2.5,-1.75
4,25%,1,1.0,1.07,1.0,None,132,114,1,9.3,0.0,0.5,1.0,0.0,1.0,15.7,2.5,0.0
5,50%,2,1.0,1.8,1.0,None,162,162,1,13.5,1.0,0.5,2.8,0.0,1.0,20.6,2.5,0.0
6,75%,2,1.0,3.4,1.0,None,234,234,1,21.2,2.5,0.5,4.34,0.0,1.0,29.8,2.5,0.0
7,max,6,9.0,342344.85,99.0,Y,265,265,5,6300.9,96.38,53.16,984.3,196.99,1.0,6304.9,2.75,1.75


### 2.3 Análise de Datas e Tempos (Anomalias Temporais)
A base de dados da TLC é famosa por ter taxímetros desconfigurados. Vamos procurar por: viagens no passado/futuro, e viagens onde o cliente desceu antes mesmo de subir (dropoff < pickup).

In [6]:
print("Limites Absolutos de Data:")
display(df_eda.select(min("tpep_pickup_datetime"), max("tpep_pickup_datetime")))

fora_2023 = df_eda.filter((col("tpep_pickup_datetime") < "2023-01-01") | (col("tpep_pickup_datetime") > "2023-12-31"))
print(f"Corridas fora do ano de 2023 (Ano Alvo): {fora_2023.count():,}")
display(fora_2023.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "total_amount").limit(5))

time_travel = df_eda.filter(col("tpep_dropoff_datetime") < col("tpep_pickup_datetime"))
print(f"Corridas com Viagem no Tempo (Dropoff anterior ao Pickup): {time_travel.count():,}")
display(time_travel.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance").limit(5))

Limites Absolutos de Data:


,min(tpep_pickup_datetime),max(tpep_pickup_datetime)
0,2001-01-01 00:06:49,2023-09-05 18:20:48


Corridas fora do ano de 2023 (Ano Alvo): 67


,tpep_pickup_datetime,tpep_dropoff_datetime,total_amount
0,2022-12-31 23:51:30,2022-12-31 23:56:06,13.50
1,2022-12-31 23:59:37,2023-01-01 00:07:28,21.38
2,2022-12-31 23:58:27,2023-01-01 00:02:21,10.88
3,2022-12-31 23:44:31,2022-12-31 23:48:47,12.96
4,2022-12-31 23:59:25,2023-01-01 00:19:21,31.33


Corridas com Viagem no Tempo (Dropoff anterior ao Pickup): 795


,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance
0,2023-01-20 13:35:00,2023-01-20 13:05:48,0.8
1,2023-01-23 10:43:58,2023-01-23 10:29:26,2.4
2,2023-01-24 11:00:00,2023-01-24 10:40:48,0.4
3,2023-02-02 13:02:23,2023-02-02 12:50:35,1.0
4,2023-02-02 13:59:20,2023-02-02 13:15:43,1.8


### 2.4 Análise Financeira (Valores Negativos e Gratuitos)
Valores negativos (`total_amount` < 0) costumam ocorrer devido a estornos de cartão de crédito ou cancelamentos da corrida. Corridas zeradas podem ser falhas do sistema.

In [7]:
sujeira_financeira = df_eda.filter(col("total_amount") <= 0)
print(f"Corridas com Faturamento Negativo ou Zerado (<= 0): {sujeira_financeira.count():,}")
display(sujeira_financeira.select("total_amount", "fare_amount", "payment_type", "passenger_count").limit(5))

Corridas com Faturamento Negativo ou Zerado (<= 0): 144,146


,total_amount,fare_amount,payment_type,passenger_count
0,-10.1,-5.1,4,1.0
1,-14.3,-9.3,4,2.0
2,-30.4,-25.4,4,1.0
3,-10.1,-5.1,2,1.0
4,-12.2,-7.2,4,2.0


### 2.5 Análise Física (Distâncias Fantasmas e Super Lotação)
Táxis de NY não levam 0 passageiros (isso seria o motorista rodando sozinho, o que não deve ser cobrado como corrida). Também não costumam levar 9 pessoas (exceto vans específicas, mas a frota amarela costuma ser carros comuns de 4-5 passageiros). Além disso, corridas com distância 0 são suspeitas.

In [8]:
passageiros_invalidos = df_eda.filter((col("passenger_count") <= 0) | (col("passenger_count") > 6))
print(f"Corridas com 0 passageiros (Viagens Fantasmas) ou excesso (> 6): {passageiros_invalidos.count():,}")
display(passageiros_invalidos.select("passenger_count", "total_amount", "trip_distance").limit(5))

distancia_zero = df_eda.filter((col("trip_distance") <= 0) & (col("total_amount") > 0))
print(f"Corridas com Distância 0, mas valor cobrado > 0: {distancia_zero.count():,}")
display(distancia_zero.select("trip_distance", "total_amount", "passenger_count").limit(5))

Corridas com 0 passageiros (Viagens Fantasmas) ou excesso (> 6): 273,593


,passenger_count,total_amount,trip_distance
0,0.0,20.85,1.9
1,0.0,16.30,1.3
2,0.0,13.80,1.0
3,0.0,19.65,2.3
4,0.0,17.80,2.1


Corridas com Distância 0, mas valor cobrado > 0: 203,635


,trip_distance,total_amount,passenger_count
0,0.0,16.12,1.0
1,0.0,16.32,1.0
2,0.0,18.50,1.0
3,0.0,8.00,1.0
4,0.0,74.00,1.0


### 2.6 Análise de Duração e Distâncias Extremas (Outliers Físicos)
Corridas que duram mais de 24 horas ou distâncias astronômicas (maiores que 100 milhas) fogem completamente do padrão de um Yellow Taxi em NY (que costuma ficar restrito aos 5 boroughs e aeroportos).

In [9]:
from pyspark.sql.functions import unix_timestamp

df_eda = df_eda.withColumn("duracao_minutos", (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60)

duracao_absurda = df_eda.filter(col("duracao_minutos") > 1440)
print(f"Corridas com duração maior que 24 horas (> 1440 minutos): {duracao_absurda.count():,}")
display(duracao_absurda.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "duracao_minutos", "trip_distance").limit(5))

distancia_absurda = df_eda.filter(col("trip_distance") > 100)
print(f"Corridas com distâncias maiores que 100 milhas: {distancia_absurda.count():,}")
display(distancia_absurda.select("trip_distance", "total_amount", "duracao_minutos").limit(5))

Corridas com duração maior que 24 horas (> 1440 minutos): 94


,tpep_pickup_datetime,tpep_dropoff_datetime,duracao_minutos,trip_distance
0,2023-01-01 18:45:20,2023-01-03 14:01:32,2596.200000,3.19
1,2023-01-01 18:45:20,2023-01-03 14:01:58,2596.633333,0.01
2,2023-01-01 18:45:20,2023-01-03 14:01:58,2596.633333,0.01
3,2023-01-05 07:21:50,2023-01-06 19:17:25,2155.583333,2.80
4,2023-01-05 07:21:50,2023-01-06 19:52:25,2190.583333,9.90


Corridas com distâncias maiores que 100 milhas: 532


,trip_distance,total_amount,duracao_minutos
0,105.55,250.00,112.400000
1,62359.52,2.50,41.083333
2,114.27,351.00,116.600000
3,105.24,400.65,180.316667
4,9680.61,61.32,43.950000


### 2.7 Validação do Escopo do Case (Jan-Mai 2023)
O case pede explicitamente para disponibilizarmos os dados de **Janeiro a Maio de 2023**. Precisamos checar quantos registros poluem nosso dataset caindo fora desse range rígido.

In [10]:
fora_escopo_case = df_eda.filter(
    (col("tpep_pickup_datetime") < "2023-01-01 00:00:00") |
    (col("tpep_pickup_datetime") >= "2023-06-01 00:00:00")
).count()
print(f"Corridas fora do range exigido pelo case (Janeiro a Maio 2023): {fora_escopo_case:,}")

Corridas fora do range exigido pelo case (Janeiro a Maio 2023): 104


### 2.8 IDs de Localização e RateCodes Inválidos
A documentação da TLC estabelece que os IDs de localização (`PULocationID` e `DOLocationID`) variam de 1 a 263. Valores 264 e 265 significam 'Unknown'. Qualquer valor fora de 1-265 é lixo sistêmico. Além disso, `RatecodeID` deve estar entre 1 e 6.

In [11]:
locais_invalidos = df_eda.filter(
    (col("pulocationid") < 1) | (col("pulocationid") > 265) |
    (col("dolocationid") < 1) | (col("dolocationid") > 265)
).count()
print(f"Corridas com Localização Inválida: {locais_invalidos:,}")

ratecodes_invalidos = df_eda.filter((col("ratecodeid") < 1) | (col("ratecodeid") > 6)).count()
print(f"Corridas com RatecodeID Inválido (diferente de 1 a 6): {ratecodes_invalidos:,}")

Corridas com Localização Inválida: 0
Corridas com RatecodeID Inválido (diferente de 1 a 6): 79,842


### 2.9 Análise de Consistência Matemática (Total Amount vs Componentes)
O `total_amount` cobrado do passageiro deve ser a soma exata de `fare_amount` + `extra` + `mta_tax` + `tip_amount` + `tolls_amount` + `improvement_surcharge` + `congestion_surcharge` + `airport_fee`. Diferenças absurdas indicam corrupção no registro.

In [12]:
from pyspark.sql.functions import abs

# Preenchemos nulos com 0 na hora de somar para evitar null propagation
soma_componentes = (
    col("fare_amount") + col("extra") + col("mta_tax") + col("tip_amount") + 
    col("tolls_amount") + col("improvement_surcharge") + 
    col("congestion_surcharge") + col("airport_fee")
)

# Identificamos corridas onde a diferença entre o total cobrado e a soma dos componentes é maior que 2 dólares (margem de tolerância)
inconsistentes_math = df_eda.filter(abs(col("total_amount") - soma_componentes) > 2)
print(f"Corridas com o cálculo do valor total corrompido: {inconsistentes_math.count():,}")

Corridas com o cálculo do valor total corrompido: 3,909,907


### 2.10 Velocidade Média Irreal (Física Dinâmica)
Cruzando distância com tempo, podemos achar a velocidade média (MPH). O limite de velocidade em NYC é majoritariamente 25 mph (chegando a 50-65 em rodovias locais). Um táxi com velocidade média > 100 mph é fisicamente improvável.

In [13]:
# Velocidade Média (MPH) = Distância / (Minutos / 60)
df_eda = df_eda.withColumn("velocidade_mph", col("trip_distance") / (col("duracao_minutos") / 60))

velocidade_absurda = df_eda.filter((col("duracao_minutos") > 0) & (col("velocidade_mph") > 100))
print(f"Corridas com velocidade média impossível (> 100 mph): {velocidade_absurda.count():,}")
display(velocidade_absurda.select("duracao_minutos", "trip_distance", "velocidade_mph").limit(5))

Corridas com velocidade média impossível (> 100 mph): 8,588


,duracao_minutos,trip_distance,velocidade_mph
0,0.266667,5.3,1192.500000
1,0.083333,0.8,576.000000
2,0.300000,1.2,240.000000
3,0.650000,1.2,110.769231
4,0.183333,1.2,392.727273


### 2.11 Domínio de Tipos de Pagamento
O Data Dictionary oficial da TLC diz que `payment_type` deve ser de 1 a 6 (1=Credit card, 2=Cash, 3=No charge, 4=Dispute, 5=Unknown, 6=Voided trip). Valores nulos ou fora desse range são lixo.

In [14]:
payment_invalido = df_eda.filter(~col("payment_type").isin([1, 2, 3, 4, 5, 6])).count()
print(f"Corridas com tipo de pagamento inexistente: {payment_invalido:,}")

Corridas com tipo de pagamento inexistente: 428,665


### 2.12 Conclusão da EDA e Checklist Definitivo de Limpeza

Com base na nossa exploração, mapeamos diversas anomalias. No entanto, o case do iFood exige estritamente 5 colunas (`VendorID`, `passenger_count`, `total_amount`, `tpep_pickup_datetime`, `tpep_dropoff_datetime`) para responder às duas perguntas de negócio de forma performática.

Portanto, dividimos o plano de limpeza em dois grupos:

#### 🔴 Regras Obrigatórias (Impactam DIRETAMENTE o Case):
A aplicação destas 5 regras é **vital** para não corromper a matemática das agregações solicitadas:
1. **Escopo do Case (Boundaries Temporais)**: Filtrar viagens para que `tpep_pickup_datetime` fique estritamente entre `2023-01-01` e `2023-05-31`. *(Evita distorções nas agregações mensais causadas por anos como 2001 e 2088).* 
2. **Filtro Financeiro**: Manter apenas `total_amount` > 0. *(Valores negativos de estornos derrubariam a média de faturamento da Pergunta 1).* 
3. **Filtro de Lotação**: Manter `passenger_count` > 0 e <= 6. *(Viagens fantasmas ou valores superlotados distorceriam a média de lotação da Pergunta 2).* 
4. **Sanidade da Viagem**: Filtrar `tpep_dropoff_datetime` > `tpep_pickup_datetime`. *(Registros onde o dropoff é anterior ao pickup comprometem agregações lógicas de tempo).* 
5. **Drop Nulls**: Remover linhas onde qualquer uma das 5 colunas do escopo seja nula.

#### 🟡 Regras Opcionais/Metadados (Boas Práticas de Governanca):
Anomalias detectadas em colunas que **não serão consumidas** pelo case, mas que indicam registros corrompidos na base da TLC:
- Metadados Inválidos (`PULocationID`, `DOLocationID` fora de 1-265, `RatecodeID` e `payment_type` fora de domínio).
- Velocidade Irreal (`velocidade_mph` > 100) e distâncias absurdas (`trip_distance` > 100).
- Divergência Matemática entre componentes da tarifa (`fare_amount`, `mta_tax`, etc) e o `total_amount`.

*Essa exploração finaliza nossa etapa de EDA. O script de negócio aplicará apenas o primeiro bloco de regras.*